In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. ЗАГРУЗКА ДАННЫХ
data = pd.read_parquet('../data/train_optimized.parquet')
data_last_year = data[data['date'] >= pd.Timestamp('2016-08-15')].copy()
x_test = pd.read_parquet('../data/output_test_optimized.parquet')

data_oil = pd.read_parquet('../data/output_oil_optimized.parquet')
data_oil = data_oil.sort_values('date')

# Исправлены названия колонок в соответствии с параметрами (21 и 28)
data_oil['dcoilwtico'] = data_oil['dcoilwtico'].ffill().bfill()
data_oil['oil_ma_21'] = data_oil['dcoilwtico'].rolling(window=21, min_periods=1).mean()
data_oil['oil_std_21'] = data_oil['dcoilwtico'].rolling(window=21, min_periods=1).std()
data_oil['oil_lag_28'] = data_oil['dcoilwtico'].shift(28)

data_items = pd.read_parquet('../data/output_items.parquet')
data_items['perishable'] = data_items['perishable'].astype('int8')
data_items['item_weight'] = np.where(data_items['perishable'] == 1, 1.25, 1.0).astype('float32')

# 2. СКЛЕЙКА И БАЗОВАЯ ПОДГОТОВКА
all_data = pd.concat([data_last_year, x_test], ignore_index=True)
all_data['unit_sales'] = all_data['unit_sales'].astype('float32')
all_data = all_data.sort_values(by=['store_nbr', 'item_nbr', 'date'])

# 3. ФИЧИ ИЗ ДАТЫ (Делаем СРАЗУ для всего датасета)
all_data['date'] = pd.to_datetime(all_data['date'])
all_data['year'] = all_data['date'].dt.year.astype('int16')
all_data['month'] = all_data['date'].dt.month.astype('int8')
all_data['day'] = all_data['date'].dt.day.astype('int8')
all_data['day_of_week'] = all_data['date'].dt.dayofweek.astype('int8')

# 4. ЛАГИ И СКОЛЬЗЯЩИЕ СРЕДНИЕ
all_data['lag_16'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(16).astype('float32')

# Оптимизированный расчет скользящих окон (работает в разы быстрее, чем lambda)
gb = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16']
all_data['mean_2'] = gb.transform(lambda x: x.rolling(2).mean()).astype('float32') # lambda тут оставлена, если pandas ругается на прямой rolling.transform
all_data['mean_4'] = gb.transform(lambda x: x.rolling(4).mean()).astype('float32')
all_data['mean_6'] = gb.transform(lambda x: x.rolling(6).mean()).astype('float32')
all_data['std_2']  = gb.transform(lambda x: x.rolling(2).std()).astype('float32')
all_data['std_4']  = gb.transform(lambda x: x.rolling(4).std()).astype('float32')
all_data['std_6']  = gb.transform(lambda x: x.rolling(6).std()).astype('float32')

all_data['lag_21'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(21).astype('float32')
all_data['lag_28'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(28).astype('float32')

# 5. МЕРДЖИ
all_data = all_data.merge(data_oil, on='date', how='left')
all_data = all_data.merge(data_items, on='item_nbr', how='left')

# Обработка промо (тоже для всей таблицы сразу)
all_data['onpromotion'] = all_data['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype('int8')

# Категориальные признаки
cat_cols = ['item_nbr', 'family', 'class', 'store_nbr']
for col in cat_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].astype('category')

# 6. СПЛИТ НА TRAIN И TEST
train_mask = (all_data['date'] < '2017-08-16') & all_data['lag_28'].notna()
test_mask = all_data['date'] >= '2017-08-16'

train_df = all_data[train_mask].copy()
test_df = all_data[test_mask].copy()

# 7. ПОДГОТОВКА X И y
# Логарифмируем таргет, предварительно убрав отрицательные значения продаж
y = np.log1p(train_df['unit_sales'].clip(lower=0)) 

# Удаляем таргет, дату и id (он пустой в трейне)
drop_cols = ['unit_sales', 'date', 'id']
X = train_df.drop(columns=drop_cols)

# Подготавливаем тестовую выборку (точно те же колонки удаляем)
X_test = test_df.drop(columns=drop_cols)
test_ids = test_df['id'].astype('int32')

# Проверка, что колонки идентичны (полезная практика)
assert list(X.columns) == list(X_test.columns), "Колонки в train и test не совпадают!"

# 8. ОБУЧЕНИЕ МОДЕЛИ
model = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# Можно явно указать модели, какие признаки категориальные
model.fit(X, y, categorical_feature=cat_cols)

# 9. ПРЕДСКАЗАНИЕ
# Возвращаем таргет в исходный масштаб экспонентой (т.к. делали log1p)
predictions = np.expm1(model.predict(X_test))

# 10. СОХРАНЕНИЕ
submission = pd.DataFrame({
    'id': test_ids,         
    'unit_sales': predictions    
})
submission['unit_sales'] = submission['unit_sales'].clip(lower=0)
submission.to_csv('../data/submission9.csv', index=False)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.804252 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7205
[LightGBM] [Info] Number of data points in the train set: 32815810, number of used features: 24
[LightGBM] [Info] Start training from score 1.727710
